# QA Hamiltonian hyperparameter optimization

This notebook evaluates quantum-walk (QA) Hamiltonian choices on the reduced 5,377-gene graph. Every configuration is evaluated on the **same 30 reproducible 75%/25% disease-gene splits**, and metrics are averaged across those splits. This paired design ensures that differences come from the Hamiltonian rather than different train/test genes.

The implemented Hamiltonian is

$$H = sA + \alpha P_{train},$$

where $A$ is the subgraph adjacency matrix, $s\in\{+1,-1\}$ controls the hopping sign, and $P_{train}$ is diagonal on the training genes. In `qa_score`, $\alpha$ is supplied through the `diag` argument. We begin with $\alpha=0$ and compare positive hopping (`+H`) with negative hopping (`-H`).

In [21]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from tqdm import tqdm

project_root = Path.cwd().resolve()
if project_root.name == "optimization_hyperparam":
    project_root = project_root.parents[1]
elif project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "bioGraph").is_dir():
    raise FileNotFoundError(
        "Start Jupyter from the repository root, notebooks, or "
        "notebooks/optimization_hyperparam directory."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from bioGraph.data.loading import load_disease_genes, load_ppi_graph
from bioGraph.data.splitting import split_known_genes
from bioGraph.evaluation.metrics import average_precision_at_k, recall_at_k
from bioGraph.methods.ranking import qa_score
from bioGraph.methods.utils import (
    disease_relevance_weighted_adjacency,
    scores_to_ranking,
)

In [22]:
subgraph_path = project_root / "data" / "processed" / "subgraph_5377.txt"
disease_path = project_root / "data" / "raw" / "pcbi.1004120.s004.txt"

graph = load_ppi_graph(subgraph_path)
diseases = load_disease_genes(disease_path)
nodelist = list(graph.nodes())
adjacency = sparse.csr_matrix(
    nx.adjacency_matrix(graph, nodelist=nodelist), dtype=float
)

print(f"Loaded subgraph: {graph.number_of_nodes():,} nodes, {graph.number_of_edges():,} edges")
print(f"Adjacency shape: {adjacency.shape}")

Loaded subgraph: 5,377 nodes, 94,987 edges
Adjacency shape: (5377, 5377)


## Disease-relevance-weighted adjacency

For an edge $(i,j)$, its disease-relevance score is the number of supplied disease sets containing **both** endpoint genes. The edge weight is $w_{ij}=1\cdot score_{ij}^{\beta}$. For the target disease, pass only the current training split so test genes remain hidden; all other diseases can be supplied completely. At `beta=0`, including `0**0 = 1`, this recovers the original unweighted adjacency. For `beta>0`, zero-score edges receive zero weight.

In [23]:
def visible_disease_sets(all_diseases, target_disease, train_genes):
    """Combine target training genes with complete other diseases."""
    visible = dict(all_diseases)
    visible[target_disease] = list(train_genes)
    return visible

## Experiment controls

Edit this cell to test other diseases, propagation times, diagonal strengths, hopping signs, metric cutoffs, or split counts. Keep `BASE_SEED` fixed when comparing configurations so every configuration receives identical splits.

In [32]:
DISEASE_NAME = "breast neoplasms"
TRAIN_FRACTION = 0.75
NUM_SPLITS = 30
BASE_SEED = 0

# Hyperparameter grid. Start with alpha=0 and compare hopping signs.
TIME_VALUES = [0.45]
ALPHA_VALUES = [0, 0.5]
BETA_VALUES = [0,0.5]
HOPPING_SIGNS = {"-H": -1.0}
K_VALUES = [25, 100]

## Generate the shared split schedule

These outer splits are generated once and reused by every Hamiltonian configuration. The assertions protect against overlap or loss of disease genes.

In [33]:
if DISEASE_NAME not in diseases:
    raise ValueError(f"Unknown disease: {DISEASE_NAME}")

known_genes = sorted(set(diseases[DISEASE_NAME]) & set(graph))
outer_splits = [
    split_known_genes(
        known_genes,
        train_fraction=TRAIN_FRACTION,
        random_state=BASE_SEED + split_index,
    )
    for split_index in range(NUM_SPLITS)
]

for split in outer_splits:
    assert set(split["train_genes"]).isdisjoint(split["test_genes"])
    assert set(split["train_genes"]) | set(split["test_genes"]) == set(known_genes)

print(f"Known {DISEASE_NAME} genes in subgraph: {len(known_genes)}")
print(f"Generated {len(outer_splits)} paired outer splits")
print(
    f"Split sizes: {len(outer_splits[0]['train_genes'])} train / "
    f"{len(outer_splits[0]['test_genes'])} test"
)

Known breast neoplasms genes in subgraph: 40
Generated 30 paired outer splits
Split sizes: 30 train / 10 test


## Run the QA grid over all 30 splits

For each split, all training genes initialize the walk and are excluded from the candidate ranking. Only the corresponding outer test genes count as evaluation positives.

In [34]:
records = []
total_fits = (
    len(BETA_VALUES) * len(TIME_VALUES) * len(ALPHA_VALUES)
    * len(HOPPING_SIGNS) * NUM_SPLITS
)
completed = 0

# The weighted adjacency is rebuilt inside each outer split. Only this
# split's visible target-disease training genes contribute to its score.
for split_index, split in tqdm(enumerate(outer_splits), total=len(outer_splits)):
    disease_sets_for_split = visible_disease_sets(
        diseases, DISEASE_NAME, split["train_genes"]
    )
    assert set(disease_sets_for_split[DISEASE_NAME]) == set(split["train_genes"])
    assert set(split["test_genes"]).isdisjoint(
        disease_sets_for_split[DISEASE_NAME]
    )

    for beta in BETA_VALUES:
        weighted_adjacency = disease_relevance_weighted_adjacency(
            graph, disease_sets_for_split, beta, nodelist=nodelist
        )
        if float(beta) == 0.0:
            assert (weighted_adjacency - adjacency).nnz == 0

        for alpha in ALPHA_VALUES:
            for time_value in TIME_VALUES:
                for hopping_name, hopping_sign in HOPPING_SIGNS.items():
                    hamiltonian_hopping = hopping_sign * weighted_adjacency
                    scores = qa_score(
                        graph,
                        split["train_genes"],
                        t=float(time_value),
                        H=hamiltonian_hopping,
                        diag=float(alpha),
                        nodelist=nodelist,
                    )
                    ranking = scores_to_ranking(
                        scores, nodelist, graph, split["train_genes"]
                    )
                    row = {
                        "split": split_index,
                        "hopping": hopping_name,
                        "hopping_sign": hopping_sign,
                        "alpha": float(alpha),
                        "beta": float(beta),
                        "time": float(time_value),
                    }
                    for k in K_VALUES:
                        row[f"AP@{k}"] = average_precision_at_k(
                            ranking, split["test_genes"], k=k
                        )
                        row[f"Recall@{k}"] = recall_at_k(
                            ranking, split["test_genes"], k=k
                        )
                    records.append(row)
                    completed += 1

print(f"Completed {completed}/{total_fits} QA walks")

split_results = pd.DataFrame.from_records(records)
if split_results.empty:
    raise RuntimeError("The QA grid produced no result records.")
split_results.head()

100%|██████████| 30/30 [00:47<00:00,  1.59s/it]


Completed -H, alpha=0, beta=0, t=0.45 (30/120 walks)


100%|██████████| 30/30 [00:51<00:00,  1.70s/it]


Completed -H, alpha=0.5, beta=0, t=0.45 (60/120 walks)


100%|██████████| 30/30 [00:02<00:00, 11.79it/s]


Completed -H, alpha=0, beta=0.5, t=0.45 (90/120 walks)


100%|██████████| 30/30 [00:03<00:00,  9.89it/s]

Completed -H, alpha=0.5, beta=0.5, t=0.45 (120/120 walks)


,split,hopping,hopping_sign,alpha,beta,time,AP@25,Recall@25,AP@100,Recall@100
0,0,-H,-1.0,0.0,0.0,0.45,0.000000,0.0,0.003125,0.1
1,1,-H,-1.0,0.0,0.0,0.45,0.005882,0.1,0.011011,0.2
2,2,-H,-1.0,0.0,0.0,0.45,0.000000,0.0,0.002857,0.1
3,3,-H,-1.0,0.0,0.0,0.45,0.000000,0.0,0.004082,0.2
4,4,-H,-1.0,0.0,0.0,0.45,0.000000,0.0,0.002222,0.1


## Average performance across splits

The table reports the mean and population standard deviation across the 30 paired splits. At $\alpha=0$, changing $A$ to $-A$ reverses the sign of the entire real Hamiltonian; equal probability-based results are therefore expected up to numerical precision. The paired difference table below verifies this directly.

In [35]:
# Rebuild from records so a stale pre-beta dataframe cannot be reused.
split_results = pd.DataFrame.from_records(records)
required_columns = {"split", "hopping", "alpha", "beta", "time"}
missing_columns = required_columns - set(split_results.columns)
if missing_columns:
    raise RuntimeError(
        f"Missing result columns {sorted(missing_columns)}. "
        "Rerun the QA grid cell immediately above before summarizing."
    )

metric_columns = [
    column for column in split_results.columns
    if column.startswith("AP@") or column.startswith("Recall@")
]
summary = (
    split_results
    .groupby(["hopping", "alpha", "beta", "time"], as_index=False)[metric_columns]
    .agg(["mean", "std"])
)
summary

hopping alpha beta  time     AP@25           Recall@25              AP@100  \
                                mean       std      mean       std      mean   
0      -H   0.0  0.0  0.45  0.000488  0.001510  0.010000  0.030513  0.003719   
1      -H   0.0  0.5  0.45  0.027687  0.038952  0.113333  0.081931  0.028456   
2      -H   0.5  0.0  0.45  0.000634  0.001678  0.013333  0.034575  0.003639   
3      -H   0.5  0.5  0.45  0.027978  0.038951  0.116667  0.083391  0.028490   

            Recall@100            
        std       mean       std  
0  0.002784   0.136667  0.066868  
1  0.038967   0.123333  0.085836  
2  0.002822   0.133333  0.066089  
3  0.038959   0.123333  0.085836

In [ ]:
split_results = pd.DataFrame.from_records(records)
if "beta" not in split_results:
    raise RuntimeError("Rerun the QA grid cell to generate beta-aware results.")
metric_columns = [
    column for column in split_results.columns
    if column.startswith("AP@") or column.startswith("Recall@")
]

if {"+H", "-H"} <= set(split_results["hopping"]):
    paired = split_results.pivot(
        index=["split", "alpha", "beta", "time"],
        columns="hopping",
        values=metric_columns,
    )
    paired_differences = pd.DataFrame(
        {metric: paired[(metric, "+H")] - paired[(metric, "-H")]
         for metric in metric_columns}
    )
    print("Maximum absolute paired difference (+H minus -H):")
    display(paired_differences.abs().max().to_frame("max_abs_difference"))
else:
    print("Paired sign differences require both '+H' and '-H'.")

In [ ]:
split_results = pd.DataFrame.from_records(records)
if "beta" not in split_results:
    raise RuntimeError("Rerun the QA grid cell to generate beta-aware results.")

plot_metric = f"AP@{K_VALUES[0]}"
plot_data = (
    split_results
    .groupby(["hopping", "alpha", "beta", "time"], as_index=False)[plot_metric]
    .agg(["mean", "std"])
    .reset_index()
)

hopping_values = list(plot_data["hopping"].drop_duplicates())
fig, axes = plt.subplots(
    1, len(hopping_values),
    figsize=(6 * len(hopping_values), 4),
    sharey=True,
    squeeze=False,
)

for ax, hopping_name in zip(axes.ravel(), hopping_values):
    hopping_data = plot_data[plot_data["hopping"] == hopping_name]
    for (alpha, time_value), curve in hopping_data.groupby(["alpha", "time"]):
        curve = curve.sort_values("beta")
        ax.errorbar(
            curve["beta"],
            curve["mean"],
            yerr=curve["std"].fillna(0.0),
            marker="o",
            capsize=4,
            label=f"alpha={alpha:g}, t={time_value:g}",
        )
    ax.set_xlabel("Disease-edge exponent beta")
    ax.set_title(f"Hopping {hopping_name}")
    ax.grid(alpha=0.25)
    ax.legend(title="Hamiltonian parameters", fontsize=8)

axes[0, 0].set_ylabel(f"Mean {plot_metric} over {NUM_SPLITS} splits")
fig.suptitle(f"QA disease-weighted Hamiltonian: {DISEASE_NAME}")
plt.tight_layout()
plt.show()